# segtrack quickstart

Segment, track and count objects in a video on a free Colab GPU.

Set the runtime to GPU first: **Runtime > Change runtime type > T4 GPU**.
It runs on CPU too, just slower.

Copyright (C) 2026 Bagaskara Amukti Palapa, AGPL-3.0.
Original design by [Muhammad Moin Faisal](https://github.com/MuhammadMoinFaisal);
see [NOTICE](https://github.com/BagaskaraAP-Dev/yolov8-seg-tracking/blob/main/NOTICE).

This notebook is committed with its outputs stripped, so it stays a few
kilobytes. Run the cells to fill it in.

## 1. Install

In [ ]:
!git clone --depth 1 https://github.com/BagaskaraAP-Dev/yolov8-seg-tracking.git
%cd yolov8-seg-tracking
!pip install -q -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
!segtrack --version

## 2. Add a video

Upload one of your own. Nothing is downloaded from a third party here:
the project this descends from relied on sample clips in someone's
Google Drive folder, which is a link that can disappear.

In [ ]:
from google.colab import files

uploaded = files.upload()
clip = next(iter(uploaded))
print("using:", clip)

## 3. Probe the first 30 frames

Always do this before a full run. It finishes in seconds and shows you
the frame size, so you can read off where to put the counting line.

In [ ]:
!segtrack --source "$clip" --max-frames 30 --save out/probe.mp4

In [ ]:
import cv2
from matplotlib import pyplot as plt

capture = cv2.VideoCapture("out/probe.mp4")
capture.set(cv2.CAP_PROP_POS_FRAMES, 20)
ok, frame = capture.read()
capture.release()

height, width = frame.shape[:2]
print(f"frame size: {width} x {height}")
print(f"a line across the middle would be: --line 0,{height // 2},{width},{height // 2}")

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("on")
plt.show()

## 4. Full run with counting

Edit `LINE` using the coordinates printed above, and `CLASSES` to the
objects you care about. Class names come from the model, so
`person,car,truck,bus,motorcycle` work for a COCO checkpoint.

In [ ]:
LINE = f"0,{height // 2},{width},{height // 2}"
CLASSES = "car,truck,bus,motorcycle"
MODEL = "yolov8n-seg.pt"  # yolov8x-seg.pt is more accurate and much slower

!segtrack --source "$clip" \
          --model $MODEL \
          --classes $CLASSES \
          --line $LINE \
          --save out/tracked.mp4

## 5. Download the result

Colab cannot play H.264 inline reliably, so downloading and watching it
locally is the honest option.

In [ ]:
files.download("out/tracked.mp4")

## What to try next

- `--tracker bytetrack.yaml` instead of the default BoT-SORT, then
  compare how ids survive objects passing behind each other.
- `--anchor center` instead of `bottom`, and watch when the counting
  line fires.
- `--model` pointed at your own `best.pt` from an ultralytics training
  run. Class colours are derived from the class index, so a custom
  dataset gets a usable palette with no code change.
- `docs/CARA-KERJA.md` in the repository walks through how a single
  frame travels through the pipeline, module by module.